In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 01_bronze_autoloader_dlt
# Project  : COVID-19 Clinical Data Lakehouse
# Purpose  : Ingest all raw CSV files (16 tables - Synthea 100k) into Delta
#            Tables in the bronze schema using Auto Loader within DLT.
#
# Design notes:
#   - cloudFiles.inferColumnTypes = false  => all columns are strings (Raw Truth)
#     Type casting (Date/Integer/Double) happens later in Silver after precise
#     schema definition from EDA/Profiling results.
#   - No need to manually set cloudFiles.schemaLocation or checkpointLocation
#     — DLT manages these automatically for each table.
#   - The loop uses @dlt.table as a decorator inside a separate function per
#     table (create_bronze_table) to avoid Python's Late Binding Closure issue
#     — each call creates its own scope for the table_name variable.
#   - Explicit three-level naming (catalog.schema.table): instead of relying
#     on the Pipeline Default Destination (which could be accidentally changed
#     in settings), the schema is explicitly specified in each table name — the
#     same principle applied in Silver and Gold. This keeps the entire system
#     consistent and independent of any implicit pipeline configuration.
# ============================================================

import dlt
from pyspark.sql import functions as F

catalog_name = "covid19_lakehouse"
schema_name = "bronze"
volume_path = f"/Volumes/{catalog_name}/{schema_name}/raw_files"


def create_bronze_table(table_name):
    @dlt.table(
        name=f"{catalog_name}.{schema_name}.{table_name}",
        comment=f"Raw Bronze table for {table_name} - Synthea COVID-19 100k dataset"
    )
    def generate_table():
        return (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "false")  # Raw Truth: all columns as strings
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", '"')
            .load(f"{volume_path}/{table_name}/")
            .select(
                "*",
                F.current_timestamp().alias("_ingested_at"),
                F.col("_metadata.file_name").alias("_source_file")
            )
        )
    return generate_table


table_names = [
    "patients", "encounters", "observations", "conditions",
    "medications", "procedures", "careplans", "allergies",
    "devices", "imaging_studies", "immunizations", "organizations",
    "payers", "payer_transitions", "providers", "supplies"
]

for name in table_names:
    create_bronze_table(name)